# 10 — Fine-tune DistilGPT-2 (D1 yes/no, binary toxic)

**Goal:** show that zero-shot prompting fails, but a **small** supervised adaptation makes the same decoder usable.

- Prompt: existing D1 (`Answer:`) + continuation ` yes` / ` no`
- Loss: causal LM **only on answer tokens**
- Default probe: 8k train / 1k val, 2 epochs

## Colab steps
1. Runtime → **GPU**
2. Clone/upload `DeepProjectHate` so `src/` exists
3. Run all cells
4. Download `distilgpt2-d1-ft.zip` → extract to local `models/distilgpt2-d1-ft/`
5. Local eval:
```bash
python scripts/run_model_comparison.py --models toxic-bert distilgpt2-d1 distilgpt2-d1-ft --eval-n 200
python scripts/run_fairness.py --models toxic-bert distilgpt2-d1 distilgpt2-d1-ft
```

In [ ]:
# !pip install -q transformers datasets scikit-learn pandas pyarrow accelerate

import os, sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "src").exists():
    pass
elif (ROOT / "DeepProjectHate" / "src").exists():
    ROOT = ROOT / "DeepProjectHate"
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("ROOT", ROOT.resolve())

In [ ]:
from device import get_device
print("device:", get_device())

from paths import TRAIN_CSV, SPLIT_INDICES_PATH, LOCAL_DISTILGPT2_DIR
from pathlib import Path
if not TRAIN_CSV.exists():
    %run scripts/download_jigsaw.py
if not SPLIT_INDICES_PATH.exists():
    %run scripts/make_splits.py
if not LOCAL_DISTILGPT2_DIR.exists():
    %run scripts/download_distilgpt2.py

In [ ]:
MAX_TRAIN = 8000
MAX_VAL = 1000
EPOCHS = 2
BATCH = 8  # use 4 if OOM

cmd = (
    f"python scripts/train_decoder_ft.py --epochs {EPOCHS} --batch-size {BATCH} "
    f"--max-train-samples {MAX_TRAIN} --max-val-samples {MAX_VAL}"
)
print(cmd)
!{cmd}

In [ ]:
import json
from paths import LOCAL_DISTILGPT2_D1_FT_DIR, DECODER_FT_RESULTS_PATH
print("FT dir:", LOCAL_DISTILGPT2_D1_FT_DIR, "exists=", LOCAL_DISTILGPT2_D1_FT_DIR.exists())
if DECODER_FT_RESULTS_PATH.exists():
    print(json.dumps(json.loads(DECODER_FT_RESULTS_PATH.read_text()), indent=2)[:1500])

!zip -r distilgpt2-d1-ft.zip models/distilgpt2-d1-ft
print("Download distilgpt2-d1-ft.zip → extract to models/distilgpt2-d1-ft locally.")